# AGI Phase 1.1 + 1.2 — Multi-Turn + Cross-Session (GPU repro)

Reproduces the Phase 1.1 + 1.2 demo (`experiments/agi/phase_1_1_multi_turn.py`) on a Colab GPU. The script exercises:

1. **Multi-turn chat** within a session via `ConversationManager`.
2. **LLM-driven fact extraction** with Qwen2.5-1.5B-Instruct + a few-shot prompt + a question/short-input heuristic gate (regex extractor remains as fallback).
3. **Cross-session persistence** — `AGISystem.save()` → simulated restart → `AGISystem.load()`.
4. **Privacy contract on disk** — no raw user text in the persisted JSON.

## What Phase 1.2 fixed (from the failed Phase 1.1 CPU run)

| Issue | Fix |
|---|---|
| Qwen-0.5B confabulated a full fact schema on every input | Foundation default upgraded to **Qwen2.5-1.5B-Instruct** (~3 GB; fits on T4/L4 at fp16). |
| Extraction prompt's example-list invited schema-filling | Few-shot prompt with **2 positive + 2 negative** examples and an explicit "do not invent" instruction. |
| LLM ran on questions, padded the schema | **Heuristic gate** in `LLMFactExtractor._is_question_or_too_short` skips the LLM call on inputs < 3 words or that look like questions. |
| `merge_facts` iterated `preferences` string as chars | `merge_facts` now coerces `str → [str]` before unioning. |

## Prerequisites

- **Colab Pro** with a T4, L4, or A100 GPU. Qwen2.5-1.5B is ~1.5 GB on GPU at fp16, fits on T4 (16 GB) easily.
- `Runtime → Change runtime type → GPU`.
- ~3 GB free disk for the first Qwen2.5-1.5B download (cached for subsequent runs).

## 1. Clone the repo

In [ ]:
!git clone https://github.com/frpatry/continual-synapse-layer.git
%cd continual-synapse-layer

## 2. Install pinned deps

`transformers==4.57.1` matches the `pyproject.toml` pin; `dtype=` (not `torch_dtype=`) is the load kwarg the foundation uses. Torch is preinstalled on every Colab runtime; we don't pin it here so we don't fight Colab's image.

In [ ]:
!pip install -q transformers==4.57.1

## 3. GPU verification

The foundation auto-picks CUDA + fp16 when `torch.cuda.is_available()` is True (see `src/agi/foundation.py::_pick_default_device_dtype`). If the cell below prints `CUDA available: False`, switch the runtime type to GPU before proceeding.

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Memory: {total_mem_gb:.1f} GB")
else:
    print("\n⚠️  No GPU detected — switch the runtime to GPU before continuing:")
    print("     Runtime → Change runtime type → GPU")

## 4. (Optional) Sanity check — run the AGI test suite

All AGI tests are offline (the two real-Qwen tests are gated behind `AGI_FOUNDATION_TESTS=1`). Expect **63 passed, 2 skipped** in a few seconds. Useful as a smoke test that the clone + install are intact before paying the model-download cost.

In [ ]:
!python -m pytest tests/agi/ -q

## 5. Run the Phase 1.1 + 1.2 demo

The script removes `/tmp/agi_demo_memory.json` at the top so re-runs always start from a clean state (earlier-phase runs may have left garbage extractions in that file).

What this exercises end-to-end:

- **Session 1**: 4 turns introducing the user (`Francois`, `Montréal`, `continual learning`, preference for short answers). Each turn goes through `agi.chat(...)`: observe → memory store → respond (with retrieved memory + recent turns) → log.
  - Turn 3 (`Tu peux me résumer ce qu'est l'apprentissage continu en quelques mots?`) is a question → the heuristic gate skips extraction entirely; you should see `[extraction gated: question or short input]` inline.
- **Save** the memory to `/tmp/agi_demo_memory.json`.
- **Simulated restart**: `del agi`, then `AGISystem(foundation, ...).load(...)` into a fresh instance.
- **Session 2**: 4 turns asking the assistant to recall the user (`tu te souviens de moi?`, `quel est mon nom?`, `où je vis?`, `sur quoi je travaille?`). Right answers: yes / Francois / Montréal / continual learning. The two questions get extraction-gated but **retrieval still fires** — memory lookup runs on every turn regardless of gating.
- **Privacy assertion** on the persisted JSON — programmatic check that no `raw_text` / `original_input` / `raw` / `samples` / `utterance` field is on disk.

First run downloads Qwen2.5-1.5B (~3 GB) into the runtime's HF cache; subsequent runs are model-load + 8 generation calls + 4 extraction calls (4 of the 8 turns gate out) ≈ ~2-3 min on L4.

**On the wall-time line at the bottom**: that's `/usr/bin/time -p` from the script's wrapper. Compare to the local CPU baseline (~5-10 min on Qwen-1.5B fp32).

In [ ]:
!/usr/bin/time -p python experiments/agi/phase_1_1_multi_turn.py

## 5b. Inspect the persisted JSON

What you should see:

- `state["entries"]` is a small list (one entry per Session-1 user turn that produced any facts — typically 3, since Turn 3 is gated out).
- Each entry has exactly five keys: `key`, `facts`, `timestamp`, `access_count`, `creation_session`. No raw text fields.
- `facts` reflects what the user **actually said** — `name=Francois`, `location=Montréal`, etc. — not a hallucinated schema.
- `state["current_session"]` is `1` (Session 1 was the only session active when `save()` ran).

If you see anything resembling the raw user utterances verbatim (e.g. `"Bonjour, je m'appelle Francois."`), the privacy contract has regressed and the test suite should have caught it — open an issue.

In [ ]:
import json
from pathlib import Path

save_path = Path("/tmp/agi_demo_memory.json")
state = json.loads(save_path.read_text())

print(f"current_session: {state['current_session']}")
print(f"entries: {len(state['entries'])}")
print()

forbidden = {"raw_text", "original_input", "raw", "samples", "utterance"}
for i, entry in enumerate(state["entries"]):
    print(f"--- entry {i} ---")
    print(f"  fields: {sorted(entry.keys())}")
    leaks = forbidden & set(entry.keys())
    if leaks:
        raise AssertionError(f"Privacy violation: entry {i} contains {leaks}")
    print(f"  key_dim: {len(entry['key'])}")
    print(f"  facts: {entry['facts']}")
    print(f"  creation_session: {entry['creation_session']}")
    print(f"  access_count: {entry['access_count']}")
    print()
print("✓ Privacy contract holds on disk.")

## 6. (Optional) Compare against Qwen-0.5B

Run the same demo against the previous-default `Qwen/Qwen2-0.5B-Instruct` to see the Phase 1.1 failure modes the 1.5B upgrade fixed. Uses an inline monkey-patch so we don't accidentally commit a model swap.

Expect to see (on Qwen-0.5B):
- Schema-filling on Turn 1 (`name`, `age`, `location`, `profession`, … all populated even though the user only said their name).
- Wrong answers on Session 2 — `Quel est mon nom?` → not Francois; `Où je vis?` → not Montréal.

If those failures are absent on 1.5B but present on 0.5B, the upgrade is the right Phase-1.2 lever.

In [ ]:
# Uncomment to run the same demo against Qwen-0.5B for comparison.
# ETA on L4: ~30-60s after the (smaller) 1 GB download.
#
# !/usr/bin/time -p python -c "\
# import sys; sys.path.insert(0, 'src');\
# from agi.foundation import FrozenFoundation;\
# from experiments.agi import phase_1_1_multi_turn as demo;\
# _orig = FrozenFoundation.__init__;\
# FrozenFoundation.__init__ = lambda self, model_name='Qwen/Qwen2-0.5B-Instruct', device=None, dtype=None: _orig(self, model_name=model_name, device=device, dtype=dtype);\
# raise SystemExit(demo.main())"

## 7. (Optional) Save the persisted memory + log back to Drive

Colab wipes `/tmp/` on disconnect. To keep the demo's `agi_demo_memory.json` (and your terminal log if you redirected it) across sessions, mount Drive and copy.


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/agi_phase_1_1
# !cp /tmp/agi_demo_memory.json /content/drive/MyDrive/agi_phase_1_1/